In [ ]:
import json
import requests
from groq import Groq
from dotenv import load_dotenv
import os
load_dotenv()


# Load API keys
GROQ_API_KEY = os.environ["GROQ_API_KEY"]
OPENWEATHER_API_KEY = os.environ["OPENWEATHER_API_KEY"]

client = Groq(api_key=GROQ_API_KEY)


def fetch_weather(city, units="celsius"):
    """
    Fetch current weather using FREE OpenWeather API (no subscription required)
    """

    try:
        url = "http://api.openweathermap.org/data/2.5/weather"

        params = {
            "q": city,
            "appid": OPENWEATHER_API_KEY,
            "units": "metric" if units == "celsius" else "imperial"
        }

        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        temperature = data["main"]["temp"]
        condition = data["weather"][0]["description"]

        return {
            "temperature": temperature,
            "condition": condition
        }

    except Exception as e:
        print("Error fetching weather:", e)
        return {"temperature": None, "condition": "Error"}


def get_current_weather(city, units="celsius"):
    """
    Tool wrapper for fetching weather
    """
    print(f"Fetching weather for {city}...")
    return fetch_weather(city, units)


def analyze_query(user_input):
    """
    Uses LLM to decide if the query is about weather
    and extracts city + units
    """

    prompt = f"""
You are an AI assistant.

If the user asks about weather, return JSON:

{{
  "tool_call_required": true,
  "city": "city_name",
  "units": "celsius or fahrenheit"
}}

If not weather-related:

{{
  "tool_call_required": false,
  "response": "I can only help with weather queries."
}}

User query: "{user_input}"
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"}
        )

        return json.loads(response.choices[0].message.content)

    except Exception as e:
        print("LLM error:", e)
        return {
            "tool_call_required": False,
            "response": "Error understanding request"
        }


def weather_agent(user_input):
    """
    Main logic:
    1. Understand query
    2. Call weather API if needed
    3. Return response
    """

    decision = analyze_query(user_input)

    if decision.get("tool_call_required"):
        city = decision.get("city")
        units = decision.get("units", "celsius")

        if not city:
            return "Please specify a city."

        weather = get_current_weather(city, units)

        temp = weather["temperature"]
        cond = weather["condition"]

        if temp is None:
            return f"Could not fetch weather for {city}."

        unit_symbol = "C" if units == "celsius" else "F"

        return f"The current weather in {city} is {temp}°{unit_symbol} with {cond}."

    else:
        return decision.get("response")


def run():
    """
    CLI loop
    """

    print("Weather Assistant started (type 'exit' to quit)\n")

    while True:
        user_input = input("Ask: ")

        if user_input.lower() == "exit":
            print("Goodbye!")
            break

        result = weather_agent(user_input)
        print(result)


if __name__ == "__main__":
    run()

Weather Assistant started (type 'exit' to quit)

Fetching weather for delhi...
The current weather in delhi is 30.05°C with few clouds.
Goodbye!
